In [ ]:
import base64
import json

print("Awaiting PAWN payload...")
try:
    # NOTE: Keep the literal "__PAWN_PAYLOAD_B64__" placeholder intact for the backend
    payload_raw = "__PAWN_PAYLOAD_B64__"
    
    # Adding '==' ensures proper byte alignment before decoding.
    padded_payload = payload_raw + "=="
    
    payload = json.loads(base64.b64decode(padded_payload).decode("utf-8"))
    prompt = payload.get("prompt", "a cinematic shot of a highly detailed futuristic city")
    print("Successfully decoded PAWN payload.")
except Exception as e:
    print(f"Warning: Payload decoding failed. Using default prompt. Error: {e}")
    prompt = "a cinematic shot of a highly detailed futuristic city"

print(f"Target Prompt: {prompt}")


In [ ]:
import subprocess
sys_pkg = None
try:
    import sys
    sys_pkg = sys
except ImportError:
    pass

print("Installing backend dependencies...")
packages = ["diffusers", "transformers", "accelerate"]
for pkg in packages:
    subprocess.check_call([sys_pkg.executable, "-m", "pip", "install", "-q", "--no-deps", pkg])
print("Dependencies installed successfully.")


In [ ]:
import torch
import os
from diffusers import AutoPipelineForText2Image

if prompt == "warmup":
    print("Warmup prompt detected. Skipping model loading and GPU inference.")
    from PIL import Image
    img = Image.new("RGB", (768, 512), color=(70, 130, 180))
    output_dir = "/kaggle/working"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "out.png")
    img.save(output_path)
    print(f"Warmup output successfully written to {output_path}")
else:
    print("Locating mounted dataset weights...")
    input_base = "/kaggle/input"
    target_dir = None

    # Dynamically find the model_index.json to bypass Kaggle's nested dataset folder naming
    for root, dirs, files in os.walk(input_base):
        if "model_index.json" in files:
            target_dir = root
            break

    if not target_dir:
        raise FileNotFoundError("Dataset not found! Ensure dataset_sources is correctly set in kernel-metadata.json.")

    print(f"Loading pipeline from {target_dir}...")
    pipe = AutoPipelineForText2Image.from_pretrained(
        target_dir,
        torch_dtype=torch.float16,
        use_safetensors=True,
        local_files_only=True
)
    pipe = pipe.to("cuda")
    pipe.set_progress_bar_config(disable=True)

    print("Executing inference...")
    image = pipe(
        prompt=prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
        height=1024,
        width=1024
).images[0]

    output_dir = "/kaggle/working"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "out.png")
    image.save(output_path)
    print(f"Output successfully written to {output_path} for PAWN backend retrieval.")
